# 05n: Clean Messy CSV Data with pandas and SQLite

**Time:** 30–40 minutes. **Resources:** CPU, less than 1 GB RAM. Turn a small messy sales extract into an auditable clean table, then verify that pandas and SQL compute the same summary.

**Objectives:** declare missing-value and duplicate policies, normalize timestamps and units, quarantine rejected rows, and test two independent aggregation paths.

**Prerequisites:** complete [the Jupyter introduction](05_Running_Jupyter_Lab.ipynb), select a Python 3.10+ kernel, and follow the [support guide](README.md). Packages: NumPy, pandas, scikit-learn and Matplotlib. Use your approved environment; this notebook does not install packages. CPU only, no GPU, credentials, gateway, or external service required. Run cells in order, then restart the kernel and Run All to check reproducibility.

SQLite is part of Python's standard library. All records are **invented educational examples** embedded below. No user data, database service, uploads or credentials are involved.

In [ ]:
from pathlib import Path
import sys
# Accept a kernel launched at the repository root or in this lesson directory.
cwd = Path.cwd().resolve()
LAB_DIR = cwd if (cwd / "05_Running_Jupyter_Lab.ipynb").is_file() else cwd / "05_Jupyter"
if not LAB_DIR.is_dir():
    raise RuntimeError("Start the kernel from the repository root or 05_Jupyter directory.")
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Read raw strings before making decisions

Reading as strings preserves evidence such as whitespace and invalid numbers. Keep `raw` unchanged for audit. This is a tiny sales ledger with signed quantities and prices in either dollars or cents. For this exercise, negative quantities are invalid rather than returns.

**Policy:** trim identifiers and normalize product/unit case; blank, `NA` and `null` quantities become missing; require positive integral quantities; require finite, nonnegative prices in `usd` or `cents`; convert to integer cents and reject fractional cents. Timestamp strings must contain an explicit UTC offset (`Z` or ±HH:MM). Ambiguous naive timestamps are rejected, not guessed. Convert aware times to UTC and aggregate by **UTC day**, which can differ from the local calendar day. Blank products become `unknown`, retained as an explicit category.

In [ ]:
from io import StringIO
from contextlib import closing
import sqlite3
from decimal import Decimal, InvalidOperation

CSV_TEXT = """order_id,timestamp,product,quantity,price,unit
A01,2025-06-01T10:00:00-04:00, Widget ,2,12.50,USD
A02,2025-06-01T23:30:00-04:00,gadget,1,999,cents
A03,2025-06-02T12:00:00Z,widget,NA,10,usd
A04,2025-06-02T13:00:00Z,widget,3,ten,usd
A01,2025-06-01T10:00:00-04:00, Widget ,2,12.50,USD
A05,2025-06-02T14:00:00Z,gadget,-1,8,usd
A06,2025-06-02 15:00:00,widget,1,11,usd
A07,2025-06-02T16:00:00Z,,2,5,usd
A08,2025-06-03T09:00:00+02:00,WIDGET, 3 ,1250,cents
A09,not-a-date,gadget,1,9,usd
A10,2025-06-03T11:00:00Z,gadget,2,4,eur
A11,2025-06-03T12:00:00Z,gadget,1.5,9,usd
A12,2025-06-03T13:00:00Z,widget,1,1.001,usd
A13,2025-06-03T14:00:00Z,widget,1,7,usd
A13,2025-06-03T14:00:00Z,widget,2,7,usd
"""
raw = pd.read_csv(StringIO(CSV_TEXT), dtype=str, keep_default_na=False)
raw.insert(0, "source_row", np.arange(2, len(raw) + 2))
print("Raw rows:", len(raw))
raw.head()

## 2. Resolve duplicates without hiding conflicts

An exact duplicate after trimming/case normalization is removed, keeping the first occurrence. If the same order ID has conflicting nonduplicate records, **quarantine every conflicting record** rather than choosing a convenient last value. The source-row column traces each decision to the original CSV line. In a real ledger, confirm the business key and correction/version policy with its owner before deduplicating.

In [ ]:
work = raw.copy(deep=True)
for column in work.columns.drop("source_row"):
    work[column] = work[column].str.strip()
for column in ["product", "unit"]:
    work[column] = work[column].str.lower()
work["product"] = work["product"].replace("", "unknown")
record_columns = list(work.columns.drop("source_row"))
exact_duplicate = work.duplicated(subset=record_columns, keep="first")
duplicates = work.loc[exact_duplicate].copy()
work = work.loc[~exact_duplicate].copy()
conflicting_id = work.order_id.duplicated(keep=False)
work["reason"] = ""

def reject(mask, reason):
    # Multiple reasons are preserved rather than overwritten.
    work.loc[mask, "reason"] = work.loc[mask, "reason"] + reason + "; "

reject(conflicting_id, "conflicting order_id")
reject(work.order_id.eq(""), "missing order_id")
print("Exact duplicates removed:", len(duplicates))
print("Conflicting records quarantined:", int(conflicting_id.sum()))

In [ ]:
quantity_text = work.quantity.replace({"": None, "NA": None, "null": None})
quantity = pd.to_numeric(quantity_text, errors="coerce")
quantity_ok = quantity.notna() & np.isfinite(quantity) & (quantity > 0) & (quantity % 1 == 0)
reject(~quantity_ok, "quantity must be a positive integer")

def price_to_cents(price, unit):
    if unit not in {"usd", "cents"}:
        return None
    try:
        amount = Decimal(price) * (100 if unit == "usd" else 1)
        if not amount.is_finite() or amount < 0 or amount != amount.to_integral_value():
            return None
        # Bound this teaching example to keep SQLite integer arithmetic exact.
        return int(amount) if amount <= 1_000_000 else None
    except InvalidOperation:
        return None

price_cents = pd.Series([price_to_cents(p, u) for p, u in zip(work.price, work.unit)], index=work.index, dtype="Int64")
reject(price_cents.isna(), "invalid price or currency unit")
reject(quantity > 1_000_000, "quantity exceeds lab bound")
# Parse individually: supports mixed offset representations on pandas 1.x and 2.x.
has_offset = work.timestamp.str.contains(r"(?:Z|[+-]\d{2}:\d{2})$", regex=True)
parsed_time = pd.to_datetime(work.timestamp.map(lambda text: pd.to_datetime(text, utc=True, errors="coerce")), utc=True)
reject(~has_offset | parsed_time.isna(), "invalid timestamp or missing timezone")

rejected = work.loc[work.reason.ne("")].copy()
keep = work.reason.eq("")
clean = work.loc[keep, ["source_row", "order_id", "product"]].copy()
clean["timestamp_utc"] = parsed_time.loc[keep]
clean["utc_day"] = clean.timestamp_utc.dt.strftime("%Y-%m-%d")
clean["quantity"] = quantity.loc[keep].astype("int64")
clean["price_cents"] = price_cents.loc[keep].astype("int64")
clean["revenue_cents"] = clean.quantity * clean.price_cents
assert clean.order_id.is_unique
assert len(raw) == len(duplicates) + len(rejected) + len(clean)
assert str(clean.timestamp_utc.dt.tz) == "UTC"
print("Kept:", len(clean), "Rejected:", len(rejected), "Duplicates:", len(duplicates))
print(rejected[["source_row", "order_id", "reason"]].to_string(index=False))
clean

**Checkpoint:** A02 moves to June 2 in UTC. A06 lacks an offset and must be rejected. Both conflicting A13 records are quarantined. Explain why multiplying binary floating-point dollars is a poor default for financial totals.

## 3. Compute the same answer twice

The pandas path sums the precomputed integer revenue column. The SQL path independently multiplies quantity by price before aggregation. Both use the same cleaned rows, grouping keys and UTC-day definition. SQLite runs entirely in memory and is closed automatically after the query. Agreement verifies aggregation consistency, not whether our business policy was correct.

In [ ]:
pandas_summary = (clean.groupby(["utc_day", "product"], as_index=False)
                  .agg(orders=("order_id", "size"), units=("quantity", "sum"), revenue_cents=("revenue_cents", "sum"))
                  .sort_values(["utc_day", "product"]).reset_index(drop=True))
with closing(sqlite3.connect(":memory:")) as connection:
    clean[["order_id", "utc_day", "product", "quantity", "price_cents"]].to_sql("orders", connection, index=False)
    sql_summary = pd.read_sql_query("""
        SELECT utc_day, product, COUNT(*) AS orders, SUM(quantity) AS units,
               SUM(quantity * price_cents) AS revenue_cents
        FROM orders
        GROUP BY utc_day, product
        ORDER BY utc_day, product
    """, connection)
pd.testing.assert_frame_equal(pandas_summary, sql_summary, check_dtype=False)
assert int(pandas_summary.revenue_cents.sum()) == 8249  # Hand-check the four retained orders.
assert len(clean) == 4 and len(duplicates) == 1 and len(rejected) == 10
print("pandas and SQLite agree exactly; total revenue: 8249 cents ($82.49).")
pandas_summary

## 4. Extend the policy, then save and stop

**Exercise:** add one explicit-offset order near midnight and one exact duplicate. Predict the UTC bucket and totals before running. Then propose a separate returns policy for negative quantities, including how refunds relate to original orders. Do not simply remove the positivity check without deciding the accounting meaning.

**Limitations:** this tiny CSV is not a general financial-processing system. Its fixed currency vocabulary, bounded values, source-row identifiers, rejection policy and UTC reporting day are intentional teaching choices. A real pipeline needs schema/version management, idempotence, locale and currency rules, access controls, a durable audit log and owner-approved correction rules. Do not paste private records into the example.

**Troubleshooting:** expected-total assertions should fail if you intentionally change the input; recalculate by hand before updating them. A pandas/SQL mismatch calls for checking sort order, null handling and integer types. If the timezone accessor fails, restart and run all cells so the parsing step has run. Missing packages: use the support guide's approved environment.

**Save and cleanup:** save your notebook and policy notes in the confirmed persistent workspace. The CSV and database stayed in memory; no exported dataset needs deletion. Optional exports should go to the repository's ignored `lab_outputs/` location, not tracked source files. Clear outputs if appropriate, shut down the kernel and stop the hosted session in its dashboard. Verify persistence using the introduction's save/restore process.

[Return to the section index](README.md) · [Jupyter introduction](05_Running_Jupyter_Lab.ipynb)